# CockroachDB Document Store + Index Store


`CockroachDBDocumentStore` and `CockroachDBIndexStore` are thin wrappers over `CockroachDBKVStore` (sync + async). They use the `cockroachdb+psycopg2` and `cockroachdb+asyncpg` SQLAlchemy dialects, so transactions retry transparently on `SERIALIZATION_FAILURE`. Pair them with `CockroachDBVectorStore` for a fully CRDB-backed `StorageContext`.


## Setup

In [ ]:
%pip install llama-index llama-index-cockroachdb

Start a local single-node insecure cluster (development only):

```bash
cockroach start-single-node --insecure --listen-addr=localhost:26257
```


## Build the stores

In [ ]:
from llama_index.storage.docstore.cockroachdb import CockroachDBDocumentStore
from llama_index.storage.index_store.cockroachdb import CockroachDBIndexStore

conn = dict(
    host="localhost", port=26257, database="defaultdb",
    user="root", password="", sslmode="disable",
)

docstore = CockroachDBDocumentStore.from_params(**conn, table_name="docs")
index_store = CockroachDBIndexStore.from_params(**conn, table_name="idx")

## Persist nodes through the document store

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

docs = [
    Document(text="CockroachDB is a distributed SQL database."),
    Document(text="C-SPANN partitions vectors across ranges for distributed ANN."),
    Document(text="Raft provides strongly consistent replication."),
]
nodes = SentenceSplitter().get_nodes_from_documents(docs)
docstore.add_documents(nodes)

print(f"persisted {len(nodes)} nodes")
fetched = docstore.get_document(nodes[0].id_)
print(fetched.get_content())

## Build a SummaryIndex backed by both stores

Pass both stores in a `StorageContext` to persist both the docstore and the index struct to CockroachDB.

In [ ]:
from llama_index.core import StorageContext, SummaryIndex

storage_context = StorageContext.from_defaults(
    docstore=docstore,
    index_store=index_store,
)
summary_index = SummaryIndex(nodes, storage_context=storage_context)
summary_index_id = summary_index.index_id

## Reload from CockroachDB

The ID alone is enough to reload across processes.

In [ ]:
from llama_index.core import load_index_from_storage

reloaded_storage = StorageContext.from_defaults(
    docstore=CockroachDBDocumentStore.from_params(**conn, table_name="docs"),
    index_store=CockroachDBIndexStore.from_params(**conn, table_name="idx"),
)
reloaded = load_index_from_storage(
    storage_context=reloaded_storage,
    index_id=summary_index_id,
)
print(reloaded)

## Async

Both stores expose async methods (`async_add_documents`, `aget_document`, `async_add_index_struct`, etc) backed by the `cockroachdb+asyncpg` dialect.


In [ ]:
await CockroachDBDocumentStore.from_params(**conn, table_name="docs_async").async_add_documents([
    Document(text="async hello world")
])

## Storage layout

Each store creates one table per `table_name` you pass, with columns `(id, key, namespace, value)`. The `value` column defaults to `JSONB` (CockroachDB's canonical JSON type) and is fully indexable via JSONB-extracted BTREE indices.
